# EC1036 - Inteligência Computacional
## Tópico: Retrieval Augmented Generation

*OBS: Esse notebook foi desenvolvido para ser executado no ambiente do google colab*

In [1]:
!pip install -q transformers datasets accelerate torch sentence-transformers faiss-cpu pypdf langchain-text-splitters pypdf

## Verificando quais disciplinas estão presentes no dataset MMLU

In [2]:
from datasets import get_dataset_config_names

print("Buscando as disciplinas disponíveis para o dataset 'cais/mmlu'...")
num_disciplinas = 0
try:
    disciplinas = get_dataset_config_names("cais/mmlu")
    print("\nDisciplinas encontradas:")
    for disciplina in disciplinas:
        num_disciplinas += 1
        print(f"- {disciplina}")
    print(f"\nTotal de disciplinas encontradas: {num_disciplinas}")
except Exception as e:
    print(f"Ocorreu um erro ao buscar as disciplinas: {e}")
    print("Certifique-se de que a biblioteca 'datasets' está instalada e que 'cais/mmlu' é um nome de dataset válido e acessível.")

Buscando as disciplinas disponíveis para o dataset 'cais/mmlu'...



Disciplinas encontradas:
- abstract_algebra
- all
- anatomy
- astronomy
- auxiliary_train
- business_ethics
- clinical_knowledge
- college_biology
- college_chemistry
- college_computer_science
- college_mathematics
- college_medicine
- college_physics
- computer_security
- conceptual_physics
- econometrics
- electrical_engineering
- elementary_mathematics
- formal_logic
- global_facts
- high_school_biology
- high_school_chemistry
- high_school_computer_science
- high_school_european_history
- high_school_geography
- high_school_government_and_politics
- high_school_macroeconomics
- high_school_mathematics
- high_school_microeconomics
- high_school_physics
- high_school_psychology
- high_school_statistics
- high_school_us_history
- high_school_world_history
- human_aging
- human_sexuality
- international_law
- jurisprudence
- logical_fallacies
- machine_learning
- management
- marketing
- medical_genetics
- miscellaneous
- moral_disputes
- moral_scenarios
- nutrition
- philosophy
- pr

## Importando bibliotecas

In [3]:
import torch
import faiss
import requests
import numpy as np
import time
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
import random


## Baixando o dataset escolhido

In [4]:
dataset = load_dataset("cais/mmlu", "astronomy", split="test").select(range(100))
print(f"{len(dataset)} questões carregadas.\n")

100 questões carregadas.



### Baixando arquivos relevantes

In [5]:
# Baixando textos relacionados na wikipedia
from langchain_core.documents import Document
titulos_wiki = [
    "Astronomy", "Astrophysics", "Solar_System", "Planet", "Exoplanet",
    "Star", "Galaxy", "Milky_Way", "Black_hole", "Nebula",
    "Supernova", "Cosmology", "Big_Bang", "Dark_matter", "Dark_energy",
    "Telescope", "Hubble_Space_Telescope", "James_Webb_Space_Telescope",
    "Observatory", "Asteroid", "Comet", "Meteoroid",
    "Gravity", "Orbit", "Speed_of_light", "Light-year",
    "Constellation", "Interstellar_medium"
]
url_wiki = "https://en.wikipedia.org/w/api.php"
documentos_wikipedia = []
headers = {
    'User-Agent': 'ColabNotebook/1.0 (projeto_rag@exemplo.com)'
}
for titulo in titulos_wiki:
    parametros = {
        "action": "query",
        "format": "json",
        "titles": titulo,
        "prop": "extracts",
        "explaintext": True,
    }
    try:
        resposta = requests.get(url_wiki, params=parametros, headers=headers)
        if resposta.status_code == 200:
            dados_wiki = resposta.json()
            paginas = dados_wiki["query"]["pages"]
            texto_extraido = list(paginas.values())[0].get("extract", "")
            print(f"titulo baixado:{titulo}")
            if texto_extraido:
              # Cria um objeto Document com metadados
              doc = Document(
                  page_content=texto_extraido,
                  metadata={
                  "source": "Wikipedia",
                  "title": titulo,
                  "url": f"https://en.wikipedia.org/wiki/{titulo}"
                  }
                )
              documentos_wikipedia.append(doc)
    except Exception as e:
        pass # Ignora silenciosamente se um artigo falhar para não poluir o terminal

print(f"Total de {len(documentos_wikipedia)} documentos criados a partir da Wikipedia.")

titulo baixado:Astronomy
titulo baixado:Astrophysics
titulo baixado:Solar_System
titulo baixado:Planet
titulo baixado:Exoplanet
titulo baixado:Star
titulo baixado:Galaxy
titulo baixado:Milky_Way
titulo baixado:Black_hole
titulo baixado:Nebula
titulo baixado:Supernova
titulo baixado:Cosmology
titulo baixado:Big_Bang
titulo baixado:Dark_matter
titulo baixado:Dark_energy
titulo baixado:Telescope
titulo baixado:Hubble_Space_Telescope
titulo baixado:James_Webb_Space_Telescope
titulo baixado:Observatory
titulo baixado:Asteroid
titulo baixado:Comet
titulo baixado:Meteoroid
titulo baixado:Gravity
titulo baixado:Orbit
titulo baixado:Speed_of_light
titulo baixado:Light-year
titulo baixado:Constellation
titulo baixado:Interstellar_medium
Total de 28 documentos criados a partir da Wikipedia.


In [6]:
documentos_wikipedia[0].page_content[:500]

'Astronomy is a natural science that studies celestial objects and the phenomena that occur in the cosmos. It uses mathematics, physics, and chemistry to explain their origin and their overall evolution. Objects of interest include planets, moons, stars, nebulae, galaxies, meteoroids, asteroids, and comets. Relevant phenomena include supernova explosions, gamma ray bursts, quasars, blazars, pulsars, and cosmic microwave background radiation. More generally, astronomy studies everything that origi'

In [7]:
documentos_wikipedia[0].metadata

{'source': 'Wikipedia',
 'title': 'Astronomy',
 'url': 'https://en.wikipedia.org/wiki/Astronomy'}

In [8]:
import requests
# Baixando textos de pdf
pdf_url = "https://www.astroshop.eu/Produktdownloads/56298_2_Leseprobe.pdf"
nome_arquivo_pdf = "exemplo_download.pdf"

try:
    print(f"Baixando PDF de: {pdf_url}")
    resposta = requests.get(pdf_url, stream=True)
    resposta.raise_for_status() # Lança um erro para códigos de status HTTP ruins (4xx ou 5xx)

    with open(nome_arquivo_pdf, 'wb') as pdf_file:
        for chunk in resposta.iter_content(chunk_size=8192):
            pdf_file.write(chunk)

    print(f"PDF baixado com sucesso e salvo como '{nome_arquivo_pdf}'")

except requests.exceptions.RequestException as e:
    print(f"Erro ao baixar o PDF: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Baixando PDF de: https://www.astroshop.eu/Produktdownloads/56298_2_Leseprobe.pdf
PDF baixado com sucesso e salvo como 'exemplo_download.pdf'


In [9]:
from pypdf import PdfReader
from langchain_core.documents import Document

documentos_pdf = [] # Alterado para uma lista de Documentos
try:
    reader = PdfReader(nome_arquivo_pdf)
    num_paginas = len(reader.pages)
    print(f"Extraindo texto de {num_paginas} páginas do PDF e criando Documentos...")
    for i, page in enumerate(reader.pages):
        texto_pagina = page.extract_text()
        if texto_pagina: # Adiciona apenas se houver texto na página
            # Cria um objeto Document com metadados para cada página
            doc = Document(
                page_content=texto_pagina,
                metadata={
                    "source": "PDF",
                    "file_name": nome_arquivo_pdf,
                    "page_number": i + 1
                }
            )
            documentos_pdf.append(doc)

    print(f"Total de {len(documentos_pdf)} documentos criados a partir do PDF.")
    # Mostra um pequeno trecho do primeiro documento extraído para verificação
    if documentos_pdf:
        print("\nPrimeiro Documento do PDF (trecho do conteúdo e metadados):")
        print("="*50)
        print(f"Conteúdo (primeiros 500 chars): {documentos_pdf[8].page_content[:1000]}...")
        print(f"Metadados: {documentos_pdf[8].metadata}")
        print("="*50)
except Exception as e:
    print(f"Erro ao extrair texto do PDF: {e}")

Extraindo texto de 47 páginas do PDF e criando Documentos...
Total de 46 documentos criados a partir do PDF.

Primeiro Documento do PDF (trecho do conteúdo e metadados):
Conteúdo (primeiros 500 chars): 19
need light. But the red light should be very dim, as even a bright red light reverses 
some of your dark adaptation. Also note that the Moon through binoculars or a 
telescope is an extremely bright object. Leave your lunar viewing for last if possi-
ble. If a local light source doesn’t allow you to become fully dark-adapted, try 
draping a dark cloth over your head and the eyepiece.
Use the technique known as averted vision for faint objects such as galaxies, 
nebulae, and globular clusters. The central area of the retina is best at detecting 
color while the areas off center are better at detecting faint light. When viewing a 
faint object, try focusing to the side of the object while concentrating on the object 
itself. You will generally be able to make out fainter detail in this 

### Limpeza de dados

In [10]:
import unicodedata
import re

def limpar_texto_para_rag(texto_bruto):
    # 1. Normalização Unicode (Padroniza caracteres e lida com codificações estranhas de PDFs)
    texto = unicodedata.normalize('NFKD', texto_bruto)

    # 2. Remoção de caracteres de controle não imprimíveis e outros artefatos de PDF.
    texto = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', texto)

    # 3. Substituir espaços não-padrão (non-breaking space, tabs, form feeds, vertical tabs) por espaço comum.
    texto = texto.replace('\xa0', ' ').replace('\t', ' ').replace('\f', ' ').replace('\v', ' ')

    # 4. Remoção de URLs (Muitas vezes não são úteis para a semântica e gastam muitos tokens)
    texto = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', texto)

    # 5. Remoção de e-mails (Para privacidade e redução de ruído)
    texto = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL REMOVIDO]', texto)

    # 6. Remoção de quebras de linha excessivas (Transforma \n\n\n\n em \n\n)
    texto = re.sub(r'\n{3,}', '\n\n', texto)

    # 7. Remoção de espaços em branco duplos/múltiplos
    texto = re.sub(r' {2,}', ' ', texto)

    # 8. Remover espaços no início e no final do texto
    texto = texto.strip()

    return texto

In [11]:
print("Aplicando limpeza de texto aos documentos da wikipedia...")
for doc in documentos_wikipedia:
    doc.page_content = limpar_texto_para_rag(doc.page_content)

# Mostra um pequeno trecho do primeiro documento PDF limpo para verificação
if documentos_wikipedia:
    print("\nPrimeiro Documento do PDF (trecho do conteúdo e metadados após limpeza):")
    print("="*50)
    print(f"Conteúdo : {documentos_wikipedia[0].page_content[:500]}...")
    print(f"Metadados: {documentos_wikipedia[0].metadata}")
    print("="*50)

Aplicando limpeza de texto aos documentos da wikipedia...

Primeiro Documento do PDF (trecho do conteúdo e metadados após limpeza):
Conteúdo : Astronomy is a natural science that studies celestial objects and the phenomena that occur in the cosmos. It uses mathematics, physics, and chemistry to explain their origin and their overall evolution. Objects of interest include planets, moons, stars, nebulae, galaxies, meteoroids, asteroids, and comets. Relevant phenomena include supernova explosions, gamma ray bursts, quasars, blazars, pulsars, and cosmic microwave background radiation. More generally, astronomy studies everything that origi...
Metadados: {'source': 'Wikipedia', 'title': 'Astronomy', 'url': 'https://en.wikipedia.org/wiki/Astronomy'}


In [12]:
print("Aplicando limpeza de texto aos documentos PDF...")
for doc in documentos_pdf:
    doc.page_content = limpar_texto_para_rag(doc.page_content)

# Mostra um pequeno trecho do primeiro documento PDF limpo para verificação
if documentos_pdf:
    print("\nPrimeiro Documento do PDF (trecho do conteúdo e metadados após limpeza):")
    print("="*50)
    print(f"Conteúdo : {documentos_pdf[6].page_content[:500]}...")
    print(f"Metadados: {documentos_pdf[6].metadata}")
    print("="*50)

Aplicando limpeza de texto aos documentos PDF...

Primeiro Documento do PDF (trecho do conteúdo e metadados após limpeza):
Conteúdo : 17In the case of deep sky objects, a better measure of luminosity is “surface brightness.” Surface brightness is not standardized and thus varies from one recorder to another, but is generally a measurement of magnitude per square arc-minute. Using such a measure we can better compare deep sky objects and deter -mine whether we should be able to view them in our telescope or binoculars. One slight complication for using surface brightness is the fact that not all objects are uniformly bright acr...
Metadados: {'source': 'PDF', 'file_name': 'exemplo_download.pdf', 'page_number': 7}


## Fazendo chunking

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

# Combinar todos os documentos (Wikipedia e PDF)
documentos_completos = documentos_wikipedia + documentos_pdf

# Usar split_documents para dividir os Documentos e manter os metadados
textos_corpus = text_splitter.split_documents(documentos_completos)

print(f"Documentos processados! Divididos em {len(textos_corpus)} blocos de textos menores (chunks)")

# Opcional: Mostrar um exemplo de um chunk com seus metadados
if textos_corpus:
    print("\nExemplo de um chunk (trecho do conteúdo e metadados):")
    print("="*50)
    print(f"Conteúdo (primeiros 200 chars): {textos_corpus[0].page_content[:200]}...")
    print(f"Metadados: {textos_corpus[0].metadata}")
    print("="*50)

Documentos processados! Divididos em 3291 blocos de textos menores (chunks)

Exemplo de um chunk (trecho do conteúdo e metadados):
Conteúdo (primeiros 200 chars): Astronomy is a natural science that studies celestial objects and the phenomena that occur in the cosmos. It uses mathematics, physics, and chemistry to explain their origin and their overall evolutio...
Metadados: {'source': 'Wikipedia', 'title': 'Astronomy', 'url': 'https://en.wikipedia.org/wiki/Astronomy'}


## Baixando modelo de embeddings e Banco de dados vetorial

In [14]:
# ---------------------------------------------------------
# EMBEDDINGS (BGE-Small para alta precisão textual)
# ---------------------------------------------------------
print("Carregando modelo de Embeddings Avançado (BGE-small)...")
modelo_embedding = SentenceTransformer('BAAI/bge-small-en-v1.5')
print("Gerando embeddings para os artigos (isso leva poucos segundos)...")
embeddings_corpus = modelo_embedding.encode([doc.page_content for doc in textos_corpus], show_progress_bar=False)
dimensao_vetor = embeddings_corpus.shape[1]

# Definir o número de listas (clusters) para o IVF
nlist = 100  # Um bom ponto de partida, ajuste conforme o tamanho do seu corpus

# O quantizador é um índice Flat que particiona o espaço
quantizer = faiss.IndexFlatL2(dimensao_vetor)

# Cria o índice IVF
indice_faiss = faiss.IndexIVFFlat(quantizer, dimensao_vetor, nlist, faiss.METRIC_L2)

# O índice IVF precisa ser treinado antes de adicionar os vetores
print(f"Treinando o índice FAISS com {nlist} listas...")
indice_faiss.train(embeddings_corpus)

# Adiciona os embeddings ao índice treinado
print("Adicionando embeddings ao índice FAISS...")
indice_faiss.add(embeddings_corpus)

print(f"Banco vetorial criado com {indice_faiss.ntotal} documentos injetados usando IndexIVFFlat e {nlist} listas.\n")

Carregando modelo de Embeddings Avançado (BGE-small)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Gerando embeddings para os artigos (isso leva poucos segundos)...
Treinando o índice FAISS com 100 listas...
Adicionando embeddings ao índice FAISS...
Banco vetorial criado com 3291 documentos injetados usando IndexIVFFlat e 100 listas.



**O modelo BGE exige esta instrução antes da query para funcionar bem em retrieval**

Por que o BGE exige isso?

Isso acontece por causa da forma como o modelo foi treinado. O modelo foi ensinado a tratar "perguntas curtas" e "documentos longos" de maneiras ligeiramente diferentes para que eles se encontrem no espaço vetorial de forma mais eficiente. A instrução funciona como uma "chave" que avisa o modelo: "atenção, o texto a seguir é uma pergunta procurando por uma resposta, e não um documento comum".

Se você não usar a instrução nas queries de retrieval, o modelo fará a vetorização de forma simétrica (como se estivesse apenas medindo a similaridade semântica entre duas frases comuns), o que reduz bastante a precisão na hora de encontrar os documentos corretos.

[BGE: One-Stop Retrieval Toolkit For Search and RAG](https://github.com/FlagOpen/FlagEmbedding)

In [15]:
def buscar_no_banco_vetorial(pergunta, top_k=2, return_list=False):
    instrucao_bge = "Represent this sentence for searching relevant passages: "
    query_final = instrucao_bge + pergunta
    vetor_pergunta = modelo_embedding.encode([query_final])
    distancias, indices = indice_faiss.search(vetor_pergunta, k=top_k)
    documentos_recuperados = [textos_corpus[idx] for idx in indices[0] if idx < len(textos_corpus)]

    if return_list:
        return documentos_recuperados
    else:
        return "\n\n".join(documentos_recuperados)

### Baixando LLM pelo huggingface

In [16]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("      Modelo carregado com sucesso!\n")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

      Modelo carregado com sucesso!



### Funções de inferência e busca

In [17]:
letras = ['A', 'B', 'C', 'D']
def formatar_questao(questao):
    texto = f"Question: {questao['question']}\n\nChoices:\n"
    for i, alternativa in enumerate(questao['choices']):
        texto += f"{letras[i]}) {alternativa}\n"
    gabarito = letras[questao['answer']]
    return texto, gabarito

def extrair_alternativa(texto_gerado):
    texto = texto_gerado.upper().strip()
    match = re.search(r'(?:^|\s|\*|\"|\'|:|\()([A-D])(?:$|\s|\.|\)|:|\*|\"|\')', texto)
    if match:
        return match.group(1)
    for char in texto:
        if char in ['A', 'B', 'C', 'D']:
            return char
    return "N/A"

def obter_resposta_llm(prompt_sistema, prompt_usuario):
    mensagens = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": prompt_usuario}
    ]
    texto_formatado = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(texto_formatado, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=15, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    resposta_bruta = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    letra_extraida = extrair_alternativa(resposta_bruta)
    return resposta_bruta, letra_extraida

### Validação do RAG

In [18]:
print("Avaliando", len(dataset), "questões...\n")

acertos_baseline = 0
acertos_rag = 0

start_time = time.time()

historico_resultados = [] # Inicializa a lista para armazenar os resultados

# Define PromptTemplates outside the loop
baseline_system_template = PromptTemplate(
    template=(
        "You are an expert in Computer Security.\n"
        "Read the question and choices.\n"
        "Output ONLY the single letter of the correct choice (A, B, C, or D).\n"
    ),
    input_variables=[]
)

rag_system_template = PromptTemplate(
    template=(
        "You are an expert in Computer Security.\n"
        "Read the REFERENCE CONTEXT. If it contains the answer, use it. If it is irrelevant or unhelpful, IGNORE IT and rely on your own knowledge.\n"
        "Output ONLY the single letter of the correct choice (A, B, C, or D).\n"
    ),
    input_variables=[]
)

rag_user_prompt_template = PromptTemplate(
    template=(
        "--- REFERENCE CONTEXT ---\n"
        "{contexto_recuperado}\n"
        "-------------------------\n\n"
        "{texto_questao}"
    ),
    input_variables=["contexto_recuperado", "texto_questao"]
)

for i, linha in enumerate(dataset):
    texto_questao, gabarito = formatar_questao(linha)

    print(f"\n--- QUESTÃO {i+1} ---")
    print(f"Gabarito Correto: [{gabarito}]")

    # --- TESTE 1: ZERO-SHOT (Sem RAG) ---
    sistema_baseline = baseline_system_template.format()
    resposta_bruta_base, resposta_baseline = obter_resposta_llm(sistema_baseline, texto_questao)

    if resposta_baseline == gabarito:
        acertos_baseline += 1
        print(f"Sem RAG  -> Extraído: [{resposta_baseline}] ✅ | Original: '{resposta_bruta_base}'")
    else:
        print(f"Sem RAG  -> Extraído: [{resposta_baseline}] ❌ | Original: '{resposta_bruta_base}'")

    # --- TESTE 2: COM RAG ---

    # Query Expansion (Pergunta + Alternativas)
    query_expandida = linha['question'] + " " + " ".join(linha['choices'])
    # Chama buscar_no_banco_vetorial para retornar uma lista de Documentos
    documentos_recuperados_list = buscar_no_banco_vetorial(query_expandida, top_k=2, return_list=True)
    # Concatena o page_content de cada Documento para o prompt RAG
    contexto_recuperado_str = "\n\n".join([doc.page_content for doc in documentos_recuperados_list])

    # PROMPT PERMISSIVO: Ensina o modelo a ignorar o contexto se for ruim, mantendo a inteligência base dele.
    sistema_rag = rag_system_template.format()

    prompt_rag = rag_user_prompt_template.format(contexto_recuperado=contexto_recuperado_str, texto_questao=texto_questao)

    resposta_bruta_rag, resposta_rag = obter_resposta_llm(sistema_rag, prompt_rag)

    if resposta_rag == gabarito:
        acertos_rag += 1
        print(f"Com RAG  -> Extraído: [{resposta_rag}] ✅ | Original: '{resposta_bruta_rag}'")
    else:
        print(f"Com RAG  -> Extraído: [{resposta_rag}] ❌ | Original: '{resposta_bruta_rag}'")

    if (i + 1) % 10 == 0 or (i + 1) == len(dataset):
        print(f"      Processadas {i+1}/{len(dataset)} questões...")
    historico_resultados.append({
    "id": i,
    "pergunta": texto_questao,
    "gabarito": gabarito,
    "resp_sem_rag": resposta_baseline,
    "resp_com_rag": resposta_rag,
    "chunks_usados": [doc.page_content for doc in documentos_recuperados_list] # Agora armazena a lista de strings
    })

tempo_total = time.time() - start_time
# ---------------------------------------------------------
# RESULTADOS FINAIS
# ---------------------------------------------------------
total = len(dataset)
print("\n" + "="*50)
print(" PLACAR FINAL (COMPUTER SECURITY - MODELO 0.5B)")
print("="*50)
print(f"Acertos SEM RAG (Baseline): {acertos_baseline}/{total} ({(acertos_baseline/total)*100:.1f}%)")
print(f"Acertos COM RAG:            {acertos_rag}/{total} ({(acertos_rag/total)*100:.1f}%)")
print("="*50)
print(f"Tempo de execução: {tempo_total/60:.1f} minutos.")
print("="*50)

Avaliando 100 questões...


--- QUESTÃO 1 ---
Gabarito Correto: [A]
Sem RAG  -> Extraído: [A] ✅ | Original: 'A'
Com RAG  -> Extraído: [A] ✅ | Original: 'A'

--- QUESTÃO 2 ---
Gabarito Correto: [D]
Sem RAG  -> Extraído: [D] ✅ | Original: 'D'
Com RAG  -> Extraído: [D] ✅ | Original: 'D'

--- QUESTÃO 3 ---
Gabarito Correto: [C]
Sem RAG  -> Extraído: [A] ❌ | Original: 'A'
Com RAG  -> Extraído: [D] ❌ | Original: 'D'

--- QUESTÃO 4 ---
Gabarito Correto: [C]
Sem RAG  -> Extraído: [A] ❌ | Original: 'A'
Com RAG  -> Extraído: [C] ✅ | Original: 'C'

--- QUESTÃO 5 ---
Gabarito Correto: [D]
Sem RAG  -> Extraído: [D] ✅ | Original: 'D'
Com RAG  -> Extraído: [D] ✅ | Original: 'D'

--- QUESTÃO 6 ---
Gabarito Correto: [C]
Sem RAG  -> Extraído: [D] ❌ | Original: 'D'
Com RAG  -> Extraído: [D] ❌ | Original: 'D'

--- QUESTÃO 7 ---
Gabarito Correto: [B]
Sem RAG  -> Extraído: [D] ❌ | Original: 'D'
Com RAG  -> Extraído: [D] ❌ | Original: 'D'

--- QUESTÃO 8 ---
Gabarito Correto: [B]
Sem RAG  -> Extraído: [D] ❌ |

### Análsie qualitativa: o impacto do RAG

In [19]:
def analisar_impacto_rag(historico_resultados):
    casos_rag_ajudou = []
    casos_rag_atrapalhou = []

    # 1. Filtrando os cenários
    for resultado in historico_resultados:
        acertou_sem = (resultado["resp_sem_rag"] == resultado["gabarito"])
        acertou_com = (resultado["resp_com_rag"] == resultado["gabarito"])

        if not acertou_sem and acertou_com:
            # Modelo não sabia, mas o RAG trouxe a resposta certa
            casos_rag_ajudou.append(resultado)
        elif acertou_sem and not acertou_com:
            # Modelo sabia de cabeça, mas o RAG trouxe um texto que o confundiu
            casos_rag_atrapalhou.append(resultado)

    # CENÁRIO 1: O RAG melhorando a acurácia
    print(f"\n CASOS ONDE O RAG AJUDOU: {len(casos_rag_ajudou)} questões identificadas.")
    if casos_rag_ajudou:
        # Sorteia um caso aleatório para não ser sempre o mesmo se rodar de novo
        exemplo_bom = random.choice(casos_rag_ajudou)

        print("\n--- EXEMPLO DE SUCESSO (O RAG salvou o modelo) ---")
        print(f"PERGUNTA: {exemplo_bom['pergunta']}")
        print(f"GABARITO: {exemplo_bom['gabarito']}")
        print(f"SEM RAG (Alucinou/Errou): {exemplo_bom['resp_sem_rag']}")
        print(f"COM RAG (Acertou): {exemplo_bom['resp_com_rag']}")

        print("\n INVESTIGAÇÃO: Por que o modelo mudou de ideia para a resposta certa?")
        print("Trechos de texto fornecidos pelo buscador (Chunks):")
        for i, chunk_text in enumerate(exemplo_bom['chunks_usados'][:2]):
            # Pega só os primeiros 300 caracteres para não poluir muito a tela
            texto_curto = chunk_text[:300].replace('\n', ' ')
            print(f"  -> Chunk {i+1}: \"...{texto_curto}...\"")

    # CENÁRIO 2: O RAG atrapalhando
    print(f"\n\n CASOS ONDE O RAG ATRAPALHOU: {len(casos_rag_atrapalhou)} questões identificadas.")
    if casos_rag_atrapalhou:
        exemplo_ruim = random.choice(casos_rag_atrapalhou)

        print("\n--- EXEMPLO DE FALHA (O RAG induziu ao erro) ---")
        print(f"PERGUNTA: {exemplo_ruim['pergunta']}")
        print(f"GABARITO: {exemplo_ruim['gabarito']}")
        print(f"SEM RAG (Sabia de cor): {exemplo_ruim['resp_sem_rag']}")
        print(f"COM RAG (Foi confundido): {exemplo_ruim['resp_com_rag']}")

        print("\n INVESTIGAÇÃO: O que tinha no texto que enganou o modelo?")
        print("Trechos de texto fornecidos pelo buscador (Chunks):")
        for i, chunk_text in enumerate(exemplo_ruim['chunks_usados'][:2]):
            texto_curto = chunk_text.replace('\n', ' ')
            print(f"  -> Chunk {i+1}: \"...{texto_curto}...\"")

analisar_impacto_rag(historico_resultados)


 CASOS ONDE O RAG AJUDOU: 18 questões identificadas.

--- EXEMPLO DE SUCESSO (O RAG salvou o modelo) ---
PERGUNTA: Question: The so-called dark energy is a model to explain ...

Choices:
A) the radiation of black holes.
B) the mass distribution of galaxies.
C) the acceleration of the universe.
D) the microwave background of the universe.

GABARITO: C
SEM RAG (Alucinou/Errou): A
COM RAG (Acertou): C

 INVESTIGAÇÃO: Por que o modelo mudou de ideia para a resposta certa?
Trechos de texto fornecidos pelo buscador (Chunks):
  -> Chunk 1: "...In physical cosmology and astronomy, dark energy is a proposed form of energy that affects the universe on its largest scales. Its primary effect is to drive the accelerating expansion of the universe. It also slows the rate of structure formation. Assuming that the lambda-CDM model of cosmology is ..."
  -> Chunk 2: "...contains dark energy in the form of a cosmological constant. This theory suggests that only gravitationally bound systems, such as ga

- O buscador trouxe o parágrafo correto, mas o modelo (0.5B) não soube interpretar?"
- Ou o buscador trouxe o parágrafo errado (ruído) e fez o modelo mudar uma resposta que ele já sabia?"